# Project Lab 1: Image Classifiers, From Scratch and With Transfer

Notebook for **Chapter 15** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch15.html

Tip: change a number, predict the output, then run the cell again.

## 15.1 How to Run a Project in This Area

## 15.2 Project: Image Classifier With Transfer Learning and a Web Demo (learner project #9)

In [ ]:
import numpy as np

def make_image(kind, rng, size=8, noise=0.15):
    img = np.zeros((size, size))
    s = size
    if kind == "hbar":
        r = rng.integers(1, s - 1); img[r, 1:s-1] = 1
    elif kind == "vbar":
        c = rng.integers(1, s - 1); img[1:s-1, c] = 1
    elif kind == "diag":
        for i in range(1, s - 1): img[i, i] = 1
    elif kind == "anti":
        for i in range(1, s - 1): img[i, s - 1 - i] = 1
    elif kind == "box":
        img[2:s-2, 2:s-2] = 1; img[3:s-3, 3:s-3] = 0
    elif kind == "dot":
        r, c = rng.integers(1, s - 2, size=2); img[r:r+2, c:c+2] = 1
    img = img + rng.normal(0, noise, img.shape)
    return np.clip(img, 0, 1)

def make_dataset(kinds, n_per_class, seed):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for label, k in enumerate(kinds):
        for _ in range(n_per_class):
            X.append(make_image(k, rng).ravel()); y.append(label)
    X = np.array(X); y = np.array(y)
    p = rng.permutation(len(y))
    return X[p], y[p]

In [ ]:
import numpy as np
from shapes import make_dataset

def softmax(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

def init(rng, n_in, n_h, n_out):
    return {"W1": rng.normal(0, np.sqrt(2/n_in), (n_in, n_h)), "b1": np.zeros(n_h),
            "W2": rng.normal(0, np.sqrt(2/n_h), (n_h, n_out)), "b2": np.zeros(n_out)}

def forward(P, X):
    H = np.maximum(0, X @ P["W1"] + P["b1"]); return H, softmax(H @ P["W2"] + P["b2"])

def train(P, X, y, epochs, lr, freeze_features=False):
    n = len(y)
    for _ in range(epochs):
        H, p = forward(P, X)
        d = p.copy(); d[np.arange(n), y] -= 1; d /= n
        gW2 = H.T @ d; gb2 = d.sum(0)
        if not freeze_features:
            dH = (d @ P["W2"].T) * (H > 0)
            P["W1"] -= lr * (X.T @ dH); P["b1"] -= lr * dH.sum(0)
        P["W2"] -= lr * gW2; P["b2"] -= lr * gb2
    return P

def acc(P, X, y): return (forward(P, X)[1].argmax(1) == y).mean()

A = ["hbar", "vbar", "diag", "anti"]       # source task
B = ["box", "dot", "hbar"]                 # new task, few labels
Xa, ya = make_dataset(A, 200, 0)
Xb, yb = make_dataset(B, 5, 1)             # 15 images only
Xt, yt = make_dataset(B, 100, 2)           # test set
rng = np.random.default_rng(0)

Pa = train(init(rng, 64, 16, 4), Xa, ya, 300, 0.5)
Xat, yat = make_dataset(A, 100, 3)
print("source task test acc:", round(acc(Pa, Xat, yat), 3))

Ps = train(init(np.random.default_rng(1), 64, 16, 3), Xb, yb, 300, 0.5)
print("B scratch      train/test:", round(acc(Ps, Xb, yb), 3), round(acc(Ps, Xt, yt), 3))
Pt = init(np.random.default_rng(1), 64, 16, 3); Pt["W1"] = Pa["W1"].copy(); Pt["b1"] = Pa["b1"].copy()
Pt = train(Pt, Xb, yb, 300, 0.5, freeze_features=True)
print("B frozen+head  train/test:", round(acc(Pt, Xb, yb), 3), round(acc(Pt, Xt, yt), 3))
Pf = init(np.random.default_rng(1), 64, 16, 3); Pf["W1"] = Pa["W1"].copy(); Pf["b1"] = Pa["b1"].copy()
Pf = train(Pf, Xb, yb, 300, 0.05)
print("B fine-tune    train/test:", round(acc(Pf, Xb, yb), 3), round(acc(Pf, Xt, yt), 3))

In [ ]:
import torch, torch.nn as nn
from torchvision import models, datasets, transforms

tfm = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                          transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train_ds = datasets.ImageFolder("data/train", tfm)           # folder name = class name
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters(): p.requires_grad = False          # freeze backbone
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))   # new head
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
# then the usual loop: forward, CrossEntropyLoss, backward, step; evaluate on data/val

In [ ]:
import gradio as gr
def predict(img):                      # img is a PIL image
    x = tfm(img).unsqueeze(0)
    with torch.no_grad(): p = model.eval()(x).softmax(1)[0]
    return {c: float(p[i]) for i, c in enumerate(train_ds.classes)}
gr.Interface(fn=predict, inputs=gr.Image(type="pil"), outputs=gr.Label(num_top_classes=3)).launch()

## 15.3 Project: A Small Image Classifier Written From Scratch in NumPy, With Gradient Checking

In [ ]:
import numpy as np
from shapes import make_dataset

def softmax(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

def conv_forward(X, K, b):               # X (N,8,8), K (F,3,3) -> (N,F,6,6)
    N, H, W = X.shape; F = K.shape[0]; out = np.zeros((N, F, H-2, W-2))
    for i in range(H-2):
        for j in range(W-2):
            patch = X[:, i:i+3, j:j+3]
            out[:, :, i, j] = np.tensordot(patch, K, axes=([1, 2], [1, 2])) + b
    return out

def conv_backward(X, K, dout):
    N, H, W = X.shape; dK = np.zeros_like(K)
    for i in range(H-2):
        for j in range(W-2):
            patch = X[:, i:i+3, j:j+3]
            dK += np.tensordot(dout[:, :, i, j], patch, axes=([0], [0]))
    return dK, dout.sum(axis=(0, 2, 3))

def pool_forward(A):                      # (N,F,6,6) -> (N,F,3,3)
    N, F, H, W = A.shape
    blocks = A.reshape(N, F, H//2, 2, W//2, 2).transpose(0, 1, 2, 4, 3, 5).reshape(N, F, H//2, W//2, 4)
    return blocks.max(-1), blocks.argmax(-1)

def pool_backward(dout, idx, shape):
    N, F, H, W = shape
    blocks = np.zeros((N, F, H//2, W//2, 4))
    np.put_along_axis(blocks, idx[..., None], dout[..., None], axis=-1)
    return blocks.reshape(N, F, H//2, W//2, 2, 2).transpose(0, 1, 2, 4, 3, 5).reshape(shape)

def model(P, X, y):
    Z = conv_forward(X, P["K"], P["bk"][None, :])
    A = np.maximum(0, Z)
    Pl, idx = pool_forward(A)
    flat = Pl.reshape(len(X), -1)
    p = softmax(flat @ P["W"] + P["b"])
    loss = -np.log(p[np.arange(len(y)), y]).mean()
    return loss, (X, Z, A, Pl, idx, flat, p)

def grads(P, X, y):
    loss, (X, Z, A, Pl, idx, flat, p) = model(P, X, y)
    n = len(y); d = p.copy(); d[np.arange(n), y] -= 1; d /= n
    g = {"W": flat.T @ d, "b": d.sum(0)}
    dflat = d @ P["W"].T
    dA = pool_backward(dflat.reshape(Pl.shape), idx, A.shape)
    dZ = dA * (Z > 0)
    g["K"], g["bk"] = conv_backward(X, P["K"], dZ)
    return loss, g

def init(rng, F=4, n_cls=4):
    return {"K": rng.normal(0, 0.5, (F, 3, 3)), "bk": np.zeros(F),
            "W": rng.normal(0, 0.3, (F*9, n_cls)), "b": np.zeros(n_cls)}

def grad_check(P, X, y, eps=1e-5):
    _, g = grads(P, X, y); worst = 0
    for name in P:
        flatp = P[name].ravel()          # a view, so edits change P
        for i in np.random.default_rng(0).choice(flatp.size, min(5, flatp.size), replace=False):
            old = flatp[i]
            flatp[i] = old + eps; lp = model(P, X, y)[0]
            flatp[i] = old - eps; lm = model(P, X, y)[0]
            flatp[i] = old
            num = (lp - lm) / (2*eps); ana = g[name].ravel()[i]
            rel = abs(num-ana) / max(1e-12, abs(num)+abs(ana)); worst = max(worst, rel)
        print(f"  {name:3s} worst-so-far rel error {worst:.2e}")
    return worst

if __name__ == "__main__":
    kinds = ["hbar", "vbar", "diag", "box"]
    X, y = make_dataset(kinds, 5, 0); X = X.reshape(-1, 8, 8)
    P = init(np.random.default_rng(0))
    print("initial loss on tiny batch:", round(model(P, X, y)[0], 4), " ln(4) =", round(np.log(4), 4))
    print("gradient check:"); w = grad_check(P, X, y)
    assert w < 1e-5
    Xtr, ytr = make_dataset(kinds, 100, 1); Xtr = Xtr.reshape(-1, 8, 8)
    Xte, yte = make_dataset(kinds, 50, 2); Xte = Xte.reshape(-1, 8, 8)
    for ep in range(1, 61):
        loss, g = grads(P, Xtr, ytr)
        for k in P: P[k] -= 0.5 * g[k]
        if ep in (1, 10, 30, 60): print(f"epoch {ep:2d} loss {loss:.4f}")
    pred = model(P, Xte, yte)[1][-1].argmax(1)
    print("test accuracy:", (pred == yte).mean())
    print("params:", sum(v.size for v in P.values()))
    cm = np.zeros((4, 4), int)
    for t, q in zip(yte, pred): cm[t, q] += 1
    print(cm)

## 15.4 Evaluation Checklist, Error Analysis and a Debug Log

A project is not done when the training loss goes down. It is done when you can say what the model gets wrong, why, and what you would do next. This section gives you a checklist, a worked error analysis you can copy for your own data, and a realistic debug log.

In [ ]:
import numpy as np
classes = ["cat", "dog", "bird", "fish"]
# rows = true class, columns = predicted class (100 test images per class)
cm = np.array([[88,  9,  2,  1],
               [14, 82,  3,  1],
               [ 3,  2, 91,  4],
               [ 1,  0,  6, 93]])
tp = np.diag(cm)
prec = tp / cm.sum(0)
rec = tp / cm.sum(1)
f1 = 2 * prec * rec / (prec + rec)
for c, p, r, f in zip(classes, prec, rec, f1):
    print(f"{c:5s} precision {p:.3f} recall {r:.3f} F1 {f:.3f}")
print("accuracy", tp.sum() / cm.sum(), " macro F1", round(f1.mean(), 4))
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print("largest confusion:", classes[i], "predicted as", classes[j], "count", off[i, j])

# top-loss triage: the 5 most expensive test images
loss = np.array([4.1, 3.3, 3.0, 2.9, 2.2, 0.4, 0.1, 0.05])
tag  = ["mislabel", "mislabel", "blurry", "real error", "mislabel", "ok", "ok", "ok"]
top = np.argsort(-loss)[:5]
print([ (round(float(loss[k]), 1), tag[k]) for k in top])
print("share of top-5 loss that is label noise", round(sum(loss[k] for k in top if tag[k]=="mislabel")/loss[top].sum(), 3))

cat   precision 0.830 recall 0.880 F1 0.854
dog   precision 0.882 recall 0.820 F1 0.850
bird  precision 0.892 recall 0.910 F1 0.901
fish  precision 0.939 recall 0.930 F1 0.935
accuracy 0.885  macro F1 0.8849
largest confusion: dog predicted as cat count 14
[(4.1, 'mislabel'), (3.3, 'mislabel'), (3.0, 'blurry'), (2.9, 'real error'), (2.2, 'mislabel')]
share of top-5 loss that is label noise 0.619


## Chapter recap

## Practice set

**Task 1 (easy).** For a 6-class balanced problem, what loss do you expect before training? **Answer:** `ln(6) = 1.7918`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch15.html